# ⚙️ Notebook 02 — Feature Engineering & Preprocessing
### TeleConnect · Customer Churn Prediction
> **Fully automated** — just run all cells (Runtime → Run all). No manual steps needed.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install',
    'pandas==2.1.4','numpy==1.26.2','scikit-learn==1.3.2',
    'matplotlib==3.8.2','seaborn==0.13.0','imbalanced-learn==0.11.0','-q'], check=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings, os, pickle, json
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi']=100
RANDOM_STATE=42; np.random.seed(RANDOM_STATE)
os.makedirs('data/processed',exist_ok=True)
os.makedirs('models',exist_ok=True)
os.makedirs('reports/figures',exist_ok=True)
print('✅ Setup complete')

In [ ]:
# ── Load data (auto-download if not present) ──────────────────────────────────
DATA_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
try:
    df = pd.read_csv('data/raw/telco_churn_raw.csv')
    print('Loaded from local file')
except:
    df = pd.read_csv(DATA_URL)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
    os.makedirs('data/raw', exist_ok=True)
    df.to_csv('data/raw/telco_churn_raw.csv', index=False)
    print('Downloaded from URL')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

df['AvgMonthlySpend']     = np.where(df['tenure']==0, df['MonthlyCharges'], df['TotalCharges']/df['tenure'])
df['ServiceCount']        = df[service_cols].apply(lambda r: sum(v=='Yes' for v in r), axis=1)
df['ContractValue']       = df['MonthlyCharges'] * df['Contract'].map({'Month-to-month':1,'One year':12,'Two year':24})
df['ChargesRatio']        = np.where(df['TotalCharges']==0, 1.0, df['MonthlyCharges']/df['TotalCharges'])
df['IsLongTermCustomer']  = (df['tenure']>24).astype(int)

print('New features created:')
print(df[['AvgMonthlySpend','ServiceCount','ContractValue','ChargesRatio','IsLongTermCustomer']].describe().round(2))

In [ ]:
# ── Encoding ──────────────────────────────────────────────────────────────────
df_enc = df.drop(columns=['customerID','ContractValue'], errors='ignore')
# Label Encoding — binary columns (Yes/No only)
le = LabelEncoder()
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
print('Label encoding applied to:', binary_cols)

# One-Hot Encoding — multi-class nominal columns
ohe_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)
print(f'Shape after encoding: {df_enc.shape}')

In [ ]:
# ── Scaling comparison ────────────────────────────────────────────────────────
scale_cols = [c for c in ['tenure','MonthlyCharges','TotalCharges','AvgMonthlySpend','ServiceCount','ChargesRatio'] if c in df_enc.columns]
ss = StandardScaler(); mm = MinMaxScaler()
ss_vals = pd.DataFrame(ss.fit_transform(df_enc[scale_cols]), columns=[c+'_ss' for c in scale_cols])
mm_vals = pd.DataFrame(mm.fit_transform(df_enc[scale_cols]), columns=[c+'_mm' for c in scale_cols])

fig, axes = plt.subplots(1,2,figsize=(14,4))
ss_vals.boxplot(ax=axes[0]); axes[0].set_title('StandardScaler (mean=0, std=1)'); axes[0].tick_params(axis='x',rotation=45)
mm_vals.boxplot(ax=axes[1]); axes[1].set_title('MinMaxScaler (range 0-1)'); axes[1].tick_params(axis='x',rotation=45)
plt.suptitle('StandardScaler vs MinMaxScaler Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('reports/figures/02_scaler_comparison.png', bbox_inches='tight')
plt.show()
# Apply StandardScaler to final dataset
df_enc[scale_cols] = ss.fit_transform(df_enc[scale_cols])
print('✅ StandardScaler applied to final dataset')

In [ ]:
# ── Feature Selection ─────────────────────────────────────────────────────────
X = df_enc.drop(columns=['Churn']); y = df_enc['Churn']

# Method 1: Correlation
corr = X.corrwith(y).abs().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,5))
corr.head(20).plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 20 Features — Correlation with Churn'); ax.set_ylabel('|Correlation|'); ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.savefig('reports/figures/02_correlation_selection.png', bbox_inches='tight'); plt.show()

# Method 2: Mutual Information
mi = pd.Series(mutual_info_classif(X, y, random_state=RANDOM_STATE), index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,5))
mi.head(20).plot(kind='bar', ax=ax, color='darkorange', edgecolor='black')
ax.set_title('Top 20 Features — Mutual Information'); ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.savefig('reports/figures/02_mutual_info.png', bbox_inches='tight'); plt.show()

# Method 3: RFE
rfe = RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), n_features_to_select=15, step=2)
rfe.fit(X, y)
rfe_feats = X.columns[rfe.support_].tolist()

# Method 4: Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X, y)
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,5))
fi.head(20).plot(kind='bar', ax=ax, color='seagreen', edgecolor='black')
ax.set_title('Top 20 Features — Random Forest Importance'); ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.savefig('reports/figures/02_rf_importance.png', bbox_inches='tight'); plt.show()

selected_features = list(set(rfe_feats + mi.head(20).index.tolist() + fi.head(20).index.tolist()))
print(f'✅ Selected {len(selected_features)} features')

In [ ]:
# ── Class Imbalance Handling ──────────────────────────────────────────────────
X_sel = X[selected_features]
smote = SMOTE(random_state=RANDOM_STATE)
rus   = RandomUnderSampler(random_state=RANDOM_STATE)
X_sm, y_sm   = smote.fit_resample(X_sel, y)
X_rus, y_rus = rus.fit_resample(X_sel, y)

fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, (name, target) in zip(axes,[('Original',y),('SMOTE',y_sm),('Undersampled',y_rus)]):
    counts = pd.Series(target).value_counts()
    ax.bar(counts.index.astype(str), counts.values, color=['steelblue','salmon'], edgecolor='black')
    ax.set_title(f'{name}\n{dict(counts)}')
plt.suptitle('Class Balance: Original vs SMOTE vs Undersampled', fontsize=13)
plt.tight_layout()
plt.savefig('reports/figures/02_class_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Train / Val / Test Split ──────────────────────────────────────────────────
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_sel, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)
X_train, y_train = smote.fit_resample(X_tr, y_tr)

print(f'Train (after SMOTE): {X_train.shape} | Churn: {y_train.mean():.3f}')
print(f'Val:                 {X_val.shape}   | Churn: {y_val.mean():.3f}')
print(f'Test:                {X_test.shape}  | Churn: {y_test.mean():.3f}')

# ── Save all splits ───────────────────────────────────────────────────────────
X_train.to_csv('data/processed/X_train.csv', index=False)
X_val.to_csv('data/processed/X_val.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
pd.Series(y_train).to_csv('data/processed/y_train.csv', index=False)
pd.Series(y_val).to_csv('data/processed/y_val.csv', index=False)
pd.Series(y_test).to_csv('data/processed/y_test.csv', index=False)

# Regression splits (target = MonthlyCharges unscaled)
raw = pd.read_csv('data/raw/telco_churn_raw.csv')
y_reg_all = pd.to_numeric(raw['MonthlyCharges'], errors='coerce')
X_reg = df_enc.drop(columns=['Churn'], errors='ignore').loc[df_enc.index]
if 'MonthlyCharges' in X_reg.columns: X_reg = X_reg.drop(columns=['MonthlyCharges'])
X_r_tr, X_r_tmp, y_r_tr, y_r_tmp = train_test_split(X_reg, y_reg_all, test_size=0.30, random_state=RANDOM_STATE)
X_r_val, X_r_test, y_r_val, y_r_test = train_test_split(X_r_tmp, y_r_tmp, test_size=0.50, random_state=RANDOM_STATE)
X_r_tr.to_csv('data/processed/X_reg_train.csv', index=False)
X_r_val.to_csv('data/processed/X_reg_val.csv', index=False)
X_r_test.to_csv('data/processed/X_reg_test.csv', index=False)
y_r_tr.to_csv('data/processed/y_reg_train.csv', index=False)
y_r_val.to_csv('data/processed/y_reg_val.csv', index=False)
y_r_test.to_csv('data/processed/y_reg_test.csv', index=False)

with open('models/scaler.pkl','wb') as f: pickle.dump(ss,f)
with open('data/processed/selected_features.json','w') as f: json.dump(selected_features,f)
print('✅ All splits and artifacts saved!')